In [2]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import PowerTransformer

#Right-skewed data
df = pd.DataFrame({'House_Prices': [10000,120000,150000,200000,800000,2500000]})

df

,House_Prices
0,10000
1,120000
2,150000
3,200000
4,800000
5,2500000


In [6]:
#log transformer
df['Log_Prices'] = np.log1p(df['House_Prices'])

df

,House_Prices,Log_Prices
0,10000,9.210440
1,120000,11.695255
2,150000,11.918397
3,200000,12.206078
4,800000,13.592368
5,2500000,14.731802


In [7]:
#Yeo-Johnson Power Transformer
pt = PowerTransformer(method= 'yeo-johnson')
df['PowerTransformer'] = pt.fit_transform(df[['House_Prices']])

print(df)

   House_Prices  Log_Prices  PowerTransformer
0         10000    9.210440         -1.647406
1        120000   11.695255         -0.375694
2        150000   11.918397         -0.248464
3        200000   12.206078         -0.081004
4        800000   13.592368          0.783015
5       2500000   14.731802          1.569554


**Transformer pipline**

In [11]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier


In [12]:
titanic = sns.load_dataset('titanic')
titanic

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S,Second,man,True,NaN,Southampton,no,True
887,1,1,female,19.0,0,0,30.0000,S,First,woman,False,B,Southampton,yes,True
888,0,3,female,NaN,1,2,23.4500,S,Third,woman,False,NaN,Southampton,no,False
889,1,1,male,26.0,0,0,30.0000,C,First,man,True,C,Cherbourg,yes,True


In [13]:
num_features = titanic[['age','fare']]
cat_features = titanic[['embarked', 'sex']]

num_features

,age,fare
0,22.0,7.2500
1,38.0,71.2833
2,26.0,7.9250
3,35.0,53.1000
4,35.0,8.0500
...,...,...
886,27.0,13.0000
887,19.0,30.0000
888,NaN,23.4500
889,26.0,30.0000


In [14]:
cat_features

,embarked,sex
0,S,male
1,C,female
2,S,female
3,S,female
4,S,male
...,...,...
886,S,male
887,S,female
888,S,female
889,C,male


In [15]:
#Numerical pipeline: Impute -> Scale
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy = 'median')),
    ('scalar', StandardScaler())
])
num_pipeline

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scalar', StandardScaler())])

In [16]:
#Categorical pipeline
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy = 'most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown = 'ignore'))
])
num_pipeline

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scalar', StandardScaler())])

In [17]:
#Combine with ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, num_features),
    ('cat', cat_pipeline, cat_features)
])

preprocessor

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scalar', StandardScaler())]),
                                       age     fare
0    22.0   7.2500
1    38.0  71.2833
2    26.0   7.9250
3    35.0  53.1000
4    35.0   8.0500
..    ...      ...
886  27.0  13.0000
887  19.0  30.0000
888   NaN  23.4500
889  26.0  30.0000
890  32.0   7.7500

[891 rows x 2 columns]),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                     embarked     sex
0          S    male
1          C  female
2          S  female
3          S  female
4          S    male
..       ...     ...
886        S    male
887        S  female
888        S  female
889        C    male
890        Q    male

[891 rows x 2 columns])])

In [18]:
#Full Machine Learning Pipeline
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classiier', RandomForestClassifier(random_state = 42))
])

full_pipeline

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scalar',
                                                                   StandardScaler())]),
                                                        age     fare
0    22.0   7.2500
1    38.0  71.2833
2    26.0   7.9250
3    35.0  53.1000
4    35.0   8.0500
..    ...      ...
886  27.0  13.0000
887  19.0  30.0000
888   NaN  23.4500
889  26.0  30.0000
890  32.0   7.7500

[891 rows x 2 columns]),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                      embarked     sex
0          S    male
1          C  female
2          S  female
3          S  female
4          S    male
..       ...     ...
886        S    male
887        S  female
888        S  female
889        C    male
890        Q    male

[891 rows x 2 columns])])),
                ('classiier', RandomForestClassifier(random_state=42))])